
# PHASE 5 — FEATURE ENGINEERING & TARGET PREPARATION
## Borewell Sentinel India

**Phase:** 5  
**Status:** COMPLETED

**Input:** groundwater_master_processed.csv  
**Total observations:** 174,635

---

## 1. Phase Objective

The objective of this phase was to identify viable prediction targets, establish documented drinking-water screening thresholds, engineer geographical and temporal features, and prepare modelling datasets for subsequent machine learning development.

The phase establishes separate regression and classification modelling datasets while maintaining source-record traceability.

## 2. Phase Workflow

### Batch 1 — Target Feasibility Assessment

- Evaluated availability of all 27 chemical parameters.
- Identified primary and secondary regression candidates.
- Investigated state-wise and year-wise measurement availability.
- Assessed multi-target observation overlap.
- Evaluated the feasibility of core chemistry and trace-metal prediction.

### Batch 2 — Contamination Thresholds and Classification Labels

- Established a threshold reference register using BIS drinking-water screening requirements.
- Created acceptable-limit and permissible-limit exceedance labels where applicable.
- Added uranium threshold assessment.
- Added individual iron and manganese screening labels.
- Added a combined iron-manganese screening condition.
- Investigated classification class balance and reported zero measurements.

### Batch 3 — Feature Engineering

Created geographical features, including:

- Latitude and Longitude.
- Normalized state and district categories.
- State-district grouping.
- Coordinate-based spatial grid identifiers.
- Provisional spatial grouping identifiers.
- Missing-coordinate indicator.

Created temporal features, including:

- Year index.
- Month.
- Quarter.
- Day of year.
- Cyclical month encoding.
- Cyclical day-of-year encoding.

### Batch 4 — Modelling Dataset Preparation

- Prepared eight-target regression dataset.
- Prepared extended nine-target regression dataset.
- Prepared classification dataset with screening labels.
- Generated supporting target and threshold registers.
- Validated record identifiers, label values and exported files.

## 3. Modelling Principles

- Preserve original chemical measurements.
- Keep missing classification labels as missing.
- Do not invent negative classification cases.
- Maintain separate regression and classification targets.
- Exclude target measurements from the baseline geographical and temporal predictor set.
- Reserve spatial grouping information for appropriate model validation.
- Avoid using future observations when constructing temporal predictors.
- Preserve original source-record references.

---

**Next Phase: Phase 6 — Multi-Contaminant Prediction**


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

MASTER_PATH = (
    PROCESSED_DIR / "groundwater_master_processed.csv"
)

FEATURE_DIR = (
    PROJECT_ROOT / "reports" / "feature_engineering"
)

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

model_df = pd.read_csv(
    MASTER_PATH,
    low_memory=False
)

chemical_columns = [
    "Potential of Hydrogen (pH)",
    "Electric Conductivity (μS/cm)",
    "Total Dissolved Solids (mg/L)",
    "Carbonate (mg/L)",
    "Bicarbonate (mg/L)",
    "Total Alkalinity (mg/L as CaCO3)",
    "Chloride (mg/L)",
    "Nitrate N (mgN/L)",
    "Sulphate (mg/L)",
    "Phosphate(mg/L)",
    "Silica(mg/L)",
    "Fluoride (mg/L)",
    "Total Hardness (mgCaCO3/L)",
    "Calcium (mg/L)",
    "Magnesium (mg/L)",
    "Sodium (mg/L)",
    "Potassium (mg/L)",
    "Iron(mg/L)",
    "Arsenic (mg/L)",
    "Uranium(mg/L)",
    "Manganese (mg/L)",
    "Copper (mg/L)",
    "Lead (mg/L)",
    "Zinc (mg/L)",
    "Nickel (mg/L)",
    "Cadmium (mg/L)",
    "Chromium (mg/L)"
]

# Convert chemical measurements to numeric values.
for column in chemical_columns:
    model_df[column] = pd.to_numeric(
        model_df[column], errors="coerce"
    )

model_df["Latitude"] = pd.to_numeric(
    model_df["Latitude"], errors="coerce"
)

model_df["Longitude"] = pd.to_numeric(
    model_df["Longitude"], errors="coerce"
)

# Parse dates from the original sampling-date field.
model_df["_parsed_date"] = pd.to_datetime(
    model_df["Data Acquisition Time"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

model_df["_year"] = model_df["_parsed_date"].dt.year
model_df["_month"] = model_df["_parsed_date"].dt.month

print("PHASE 5 — MODELLING DATASET LOADED")
print("=" * 50)

print("Records:", len(model_df))
print("Columns:", len(model_df.columns))
print("Chemical parameters:", len(chemical_columns))
print("Valid dates:", model_df["_parsed_date"].notna().sum())
print("Source files:", model_df["_source_file"].nunique())

PHASE 5 — MODELLING DATASET LOADED
Records: 174635
Columns: 200
Chemical parameters: 27
Valid dates: 174635
Source files: 31


C:\Users\pokep\AppData\Local\Temp\ipykernel_30236\2354481217.py:79: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  model_df["_year"] = model_df["_parsed_date"].dt.year
C:\Users\pokep\AppData\Local\Temp\ipykernel_30236\2354481217.py:80: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  model_df["_month"] = model_df["_parsed_date"].dt.month


In [2]:
target_feasibility = []

for column in chemical_columns:

    values = model_df[column]

    finite_mask = values.notna() & np.isfinite(values)

    if column == "Potential of Hydrogen (pH)":

        valid_mask = (
            finite_mask
            & values.gt(0)
            & values.le(14)
        )

    else:

        valid_mask = finite_mask & values.ge(0)

    valid_values = values[valid_mask]

    available_count = int(len(valid_values))
    availability_pct = (
        available_count / len(model_df) * 100
    )

    if availability_pct >= 70:
        support = "High"

    elif availability_pct >= 30:
        support = "Moderate"

    elif availability_pct >= 10:
        support = "Limited"

    else:
        support = "Very sparse"

    target_feasibility.append({
        "Parameter": column,
        "Original_Numeric_Values": int(finite_mask.sum()),
        "Basic_Valid_Values": available_count,
        "Rejected_By_Basic_Screen": int(
            (finite_mask & ~valid_mask).sum()
        ),
        "Zero_Values": int(
            (valid_values == 0).sum()
        ),
        "Availability_Percentage": round(
            availability_pct, 2
        ),
        "Support_Level": support
    })

target_feasibility_df = pd.DataFrame(
    target_feasibility
).sort_values(
    "Availability_Percentage",
    ascending=False
).reset_index(drop=True)

display(target_feasibility_df)

,Parameter,Original_Numeric_Values,Basic_Valid_Values,Rejected_By_Basic_Screen,Zero_Values,Availability_Percentage,Support_Level
0,Electric Conductivity (μS/cm),173029,173029,0,2,99.08,High
1,Potential of Hydrogen (pH),162729,162719,10,0,93.18,High
2,Chloride (mg/L),162034,162033,1,10,92.78,High
3,Magnesium (mg/L),150422,150412,10,162,86.13,High
4,Calcium (mg/L),149855,149855,0,76,85.81,High
5,Bicarbonate (mg/L),147171,147170,1,739,84.27,High
6,Total Hardness (mgCaCO3/L),144408,144408,0,151,82.69,High
7,Sodium (mg/L),143470,143470,0,108,82.15,High
8,Sulphate (mg/L),126328,126326,2,5806,72.34,High
9,Carbonate (mg/L),120229,120229,0,104729,68.85,Moderate


In [3]:
primary_candidates = [
    "Potential of Hydrogen (pH)",
    "Electric Conductivity (μS/cm)",
    "Chloride (mg/L)",
    "Bicarbonate (mg/L)",
    "Sulphate (mg/L)",
    "Calcium (mg/L)",
    "Magnesium (mg/L)",
    "Sodium (mg/L)",
    "Total Hardness (mgCaCO3/L)"
]

support_df = model_df[
    ["State", "_year"] + primary_candidates
].copy()

# Apply basic numerical screening.
for column in primary_candidates:

    values = support_df[column]

    if column == "Potential of Hydrogen (pH)":
        support_df.loc[
            ~values.between(0.000001, 14),
            column
        ] = np.nan

    else:
        support_df.loc[
            values < 0,
            column
        ] = np.nan

state_counts = (
    support_df.groupby("State")[primary_candidates]
    .count()
)

year_counts = (
    support_df.groupby("_year")[primary_candidates]
    .count()
)

state_support_summary = pd.DataFrame({
    "Parameter": primary_candidates,
    "States_With_30plus_Observations": [
        (state_counts[column] >= 30).sum()
        for column in primary_candidates
    ],
    "Minimum_State_Support": [
        state_counts[column].min()
        for column in primary_candidates
    ],
    "Median_State_Support": [
        state_counts[column].median()
        for column in primary_candidates
    ]
})

year_support_summary = pd.DataFrame({
    "Parameter": primary_candidates,
    "Years_With_30plus_Observations": [
        (year_counts[column] >= 30).sum()
        for column in primary_candidates
    ],
    "Minimum_Yearly_Support": [
        year_counts[column].min()
        for column in primary_candidates
    ],
    "Median_Yearly_Support": [
        year_counts[column].median()
        for column in primary_candidates
    ]
})

print("STATE-LEVEL TARGET SUPPORT")
display(state_support_summary)

print("\nYEAR-LEVEL TARGET SUPPORT")
display(year_support_summary)

STATE-LEVEL TARGET SUPPORT


,Parameter,States_With_30plus_Observations,Minimum_State_Support,Median_State_Support
0,Potential of Hydrogen (pH),31,32,2894.0
1,Electric Conductivity (μS/cm),31,36,3112.0
2,Chloride (mg/L),30,16,2915.0
3,Bicarbonate (mg/L),30,28,2930.0
4,Sulphate (mg/L),30,28,1621.0
5,Calcium (mg/L),31,32,2937.0
6,Magnesium (mg/L),31,32,2939.0
7,Sodium (mg/L),30,16,2893.0
8,Total Hardness (mgCaCO3/L),31,30,2727.0



YEAR-LEVEL TARGET SUPPORT


,Parameter,Years_With_30plus_Observations,Minimum_Yearly_Support,Median_Yearly_Support
0,Potential of Hydrogen (pH),24,1,6442.5
1,Electric Conductivity (μS/cm),24,1,6912.0
2,Chloride (mg/L),24,1,6436.5
3,Bicarbonate (mg/L),24,1,5754.0
4,Sulphate (mg/L),24,1,4994.5
5,Calcium (mg/L),24,1,5755.0
6,Magnesium (mg/L),24,1,5903.0
7,Sodium (mg/L),24,1,5294.0
8,Total Hardness (mgCaCO3/L),24,1,5906.5


In [4]:
core_parameters = [
    "Potential of Hydrogen (pH)",
    "Electric Conductivity (μS/cm)",
    "Chloride (mg/L)",
    "Bicarbonate (mg/L)",
    "Sulphate (mg/L)",
    "Calcium (mg/L)",
    "Magnesium (mg/L)",
    "Sodium (mg/L)"
]

trace_parameters = [
    "Iron(mg/L)",
    "Arsenic (mg/L)",
    "Uranium(mg/L)",
    "Manganese (mg/L)",
    "Copper (mg/L)",
    "Lead (mg/L)",
    "Zinc (mg/L)",
    "Nickel (mg/L)",
    "Cadmium (mg/L)",
    "Chromium (mg/L)"
]

overlap_df = model_df[
    core_parameters + trace_parameters
].copy()

for column in overlap_df.columns:

    values = overlap_df[column]

    if column == "Potential of Hydrogen (pH)":
        overlap_df[column] = values.where(
            values.between(0.000001, 14)
        )
    else:
        overlap_df[column] = values.where(
            values >= 0
        )

core_overlap = overlap_df[core_parameters].notna().sum(axis=1)

trace_overlap = overlap_df[trace_parameters].notna().sum(axis=1)

overlap_summary = pd.DataFrame([
    {
        "Dataset_Group": "Core chemistry",
        "Parameters": len(core_parameters),
        "Records_With_At_Least_1": int((core_overlap >= 1).sum()),
        "Records_With_At_Least_4": int((core_overlap >= 4).sum()),
        "Records_With_At_Least_6": int((core_overlap >= 6).sum()),
        "Records_With_All": int((core_overlap == len(core_parameters)).sum())
    },
    {
        "Dataset_Group": "Trace metals",
        "Parameters": len(trace_parameters),
        "Records_With_At_Least_1": int((trace_overlap >= 1).sum()),
        "Records_With_At_Least_4": int((trace_overlap >= 4).sum()),
        "Records_With_At_Least_6": int((trace_overlap >= 6).sum()),
        "Records_With_All": int((trace_overlap == len(trace_parameters)).sum())
    }
])

print("MULTI-TARGET OBSERVATION OVERLAP")
display(overlap_summary)

print("\nNUMBER OF AVAILABLE PARAMETERS PER RECORD")

overlap_distribution = pd.DataFrame({
    "Available_Core_Parameters": core_overlap.value_counts().sort_index(),
    "Available_Trace_Parameters": trace_overlap.value_counts().sort_index()
}).fillna(0).astype(int)

display(overlap_distribution)

print("\nNO TARGETS HAVE BEEN REMOVED OR SELECTED PERMANENTLY.")

MULTI-TARGET OBSERVATION OVERLAP


,Dataset_Group,Parameters,Records_With_At_Least_1,Records_With_At_Least_4,Records_With_At_Least_6,Records_With_All
0,Core chemistry,8,173079,150318,146256,122713
1,Trace metals,10,42824,797,797,784



NUMBER OF AVAILABLE PARAMETERS PER RECORD


,Available_Core_Parameters,Available_Trace_Parameters
0,1556,131811
1,5104,33445
2,5752,7048
3,11905,1534
4,1386,0
5,2676,0
6,2738,0
7,20805,0
8,122713,1
9,0,12



NO TARGETS HAVE BEEN REMOVED OR SELECTED PERMANENTLY.


In [5]:
threshold_rows = [
    {
        "Parameter": "Potential of Hydrogen (pH)",
        "Unit": "pH",
        "Acceptable_Lower": 6.5,
        "Acceptable_Upper": 8.5,
        "Acceptable_Max": np.nan,
        "Permissible_Max": np.nan,
        "Rule": "Range; no relaxation",
        "Status": "Ready"
    },
    {
        "Parameter": "Total Dissolved Solids (mg/L)",
        "Unit": "mg/L",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": 500,
        "Permissible_Max": 2000,
        "Rule": "Upper limit",
        "Status": "Ready"
    },
    {
        "Parameter": "Chloride (mg/L)",
        "Unit": "mg/L",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": 250,
        "Permissible_Max": 1000,
        "Rule": "Upper limit",
        "Status": "Ready"
    },
    {
        "Parameter": "Sulphate (mg/L)",
        "Unit": "mg/L",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": 200,
        "Permissible_Max": 400,
        "Rule": "Conditional on Mg <= 30 mg/L",
        "Status": "Ready with condition"
    },
    {
        "Parameter": "Total Hardness (mgCaCO3/L)",
        "Unit": "mg/L as CaCO3",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": 200,
        "Permissible_Max": 600,
        "Rule": "Upper limit",
        "Status": "Ready"
    },
    {
        "Parameter": "Calcium (mg/L)",
        "Unit": "mg/L as Ca",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": 75,
        "Permissible_Max": 200,
        "Rule": "Upper limit",
        "Status": "Ready"
    },
    {
        "Parameter": "Magnesium (mg/L)",
        "Unit": "mg/L as Mg",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": 30,
        "Permissible_Max": 100,
        "Rule": "Upper limit",
        "Status": "Ready"
    },
    {
        "Parameter": "Total Alkalinity (mg/L as CaCO3)",
        "Unit": "mg/L as CaCO3",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": 200,
        "Permissible_Max": 600,
        "Rule": "Upper limit",
        "Status": "Ready"
    },
    {
        "Parameter": "Arsenic (mg/L)",
        "Unit": "mg/L as As",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": 0.01,
        "Permissible_Max": np.nan,
        "Rule": "No relaxation",
        "Status": "Provisional analyte match"
    },
    {
        "Parameter": "Cadmium (mg/L)",
        "Unit": "mg/L as Cd",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": 0.003,
        "Permissible_Max": np.nan,
        "Rule": "No relaxation",
        "Status": "Ready"
    },
    {
        "Parameter": "Lead (mg/L)",
        "Unit": "mg/L as Pb",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": 0.01,
        "Permissible_Max": np.nan,
        "Rule": "No relaxation",
        "Status": "Ready"
    },
    {
        "Parameter": "Chromium (mg/L)",
        "Unit": "mg/L as Cr",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": 0.05,
        "Permissible_Max": np.nan,
        "Rule": "No relaxation",
        "Status": "Ready"
    },
    {
        "Parameter": "Nickel (mg/L)",
        "Unit": "mg/L as Ni",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": 0.02,
        "Permissible_Max": np.nan,
        "Rule": "No relaxation",
        "Status": "Ready"
    },
    {
        "Parameter": "Copper (mg/L)",
        "Unit": "mg/L as Cu",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": 0.05,
        "Permissible_Max": 1.5,
        "Rule": "Upper limit",
        "Status": "Ready"
    },
    {
        "Parameter": "Zinc (mg/L)",
        "Unit": "mg/L as Zn",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": 5,
        "Permissible_Max": 15,
        "Rule": "Upper limit",
        "Status": "Ready"
    },
    {
        "Parameter": "Fluoride (mg/L)",
        "Unit": "mg/L as F",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": 1.0,
        "Permissible_Max": 1.5,
        "Rule": "Upper limit",
        "Status": "No observations"
    },
    {
        "Parameter": "Nitrate N (mgN/L)",
        "Unit": "mg/L as N",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": 45,
        "Permissible_Max": np.nan,
        "Rule": "Convert nitrate-N to nitrate",
        "Status": "No observations; unit conversion required"
    },
    {
        "Parameter": "Iron(mg/L)",
        "Unit": "mg/L as Fe",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": np.nan,
        "Permissible_Max": np.nan,
        "Rule": "Iron-manganese combined condition",
        "Status": "Deferred for joint-rule review"
    },
    {
        "Parameter": "Manganese (mg/L)",
        "Unit": "mg/L as Mn",
        "Acceptable_Lower": np.nan,
        "Acceptable_Upper": np.nan,
        "Acceptable_Max": np.nan,
        "Permissible_Max": np.nan,
        "Rule": "Iron-manganese combined condition",
        "Status": "Deferred for joint-rule review"
    }
]

threshold_registry = pd.DataFrame(threshold_rows)

display(threshold_registry)

,Parameter,Unit,Acceptable_Lower,Acceptable_Upper,Acceptable_Max,Permissible_Max,Rule,Status
0,Potential of Hydrogen (pH),pH,6.5,8.5,NaN,NaN,Range; no relaxation,Ready
1,Total Dissolved Solids (mg/L),mg/L,NaN,NaN,500.000,2000.0,Upper limit,Ready
2,Chloride (mg/L),mg/L,NaN,NaN,250.000,1000.0,Upper limit,Ready
3,Sulphate (mg/L),mg/L,NaN,NaN,200.000,400.0,Conditional on Mg <= 30 mg/L,Ready with condition
4,Total Hardness (mgCaCO3/L),mg/L as CaCO3,NaN,NaN,200.000,600.0,Upper limit,Ready
5,Calcium (mg/L),mg/L as Ca,NaN,NaN,75.000,200.0,Upper limit,Ready
6,Magnesium (mg/L),mg/L as Mg,NaN,NaN,30.000,100.0,Upper limit,Ready
7,Total Alkalinity (mg/L as CaCO3),mg/L as CaCO3,NaN,NaN,200.000,600.0,Upper limit,Ready
8,Arsenic (mg/L),mg/L as As,NaN,NaN,0.010,NaN,No relaxation,Provisional analyte match
9,Cadmium (mg/L),mg/L as Cd,NaN,NaN,0.003,NaN,No relaxation,Ready


In [6]:
threshold_rules = {
    "Potential of Hydrogen (pH)": {
        "type": "range",
        "lower": 6.5,
        "upper": 8.5
    },
    "Total Dissolved Solids (mg/L)": {
        "type": "upper",
        "acceptable": 500,
        "permissible": 2000
    },
    "Chloride (mg/L)": {
        "type": "upper",
        "acceptable": 250,
        "permissible": 1000
    },
    "Sulphate (mg/L)": {
        "type": "sulphate_conditional",
        "acceptable": 200,
        "permissible_if_magnesium_le_30": 400,
        "permissible_if_magnesium_gt_30": 200
    },
    "Total Hardness (mgCaCO3/L)": {
        "type": "upper",
        "acceptable": 200,
        "permissible": 600
    },
    "Calcium (mg/L)": {
        "type": "upper",
        "acceptable": 75,
        "permissible": 200
    },
    "Magnesium (mg/L)": {
        "type": "upper",
        "acceptable": 30,
        "permissible": 100
    },
    "Total Alkalinity (mg/L as CaCO3)": {
        "type": "upper",
        "acceptable": 200,
        "permissible": 600
    },
    "Arsenic (mg/L)": {
        "type": "no_relaxation",
        "acceptable": 0.01
    },
    "Cadmium (mg/L)": {
        "type": "no_relaxation",
        "acceptable": 0.003
    },
    "Lead (mg/L)": {
        "type": "no_relaxation",
        "acceptable": 0.01
    },
    "Chromium (mg/L)": {
        "type": "no_relaxation",
        "acceptable": 0.05
    },
    "Nickel (mg/L)": {
        "type": "no_relaxation",
        "acceptable": 0.02
    },
    "Copper (mg/L)": {
        "type": "upper",
        "acceptable": 0.05,
        "permissible": 1.5
    },
    "Zinc (mg/L)": {
        "type": "upper",
        "acceptable": 5,
        "permissible": 15
    }
}

threshold_labels = pd.DataFrame(index=model_df.index)

threshold_summary_rows = []

for column, rule in threshold_rules.items():

    values = model_df[column]

    valid_mask = (
        values.notna()
        & np.isfinite(values)
    )

    if rule["type"] == "range":

        valid_mask &= values.between(0.000001, 14)

        acceptable_exceedance = (
            (values < rule["lower"])
            | (values > rule["upper"])
        )

        permissible_exceedance = acceptable_exceedance

        unresolved_permissible = pd.Series(
            False, index=model_df.index
        )

    elif rule["type"] == "sulphate_conditional":

        valid_mask &= values >= 0

        magnesium = model_df["Magnesium (mg/L)"]

        magnesium_valid = (
            magnesium.notna()
            & np.isfinite(magnesium)
            & (magnesium >= 0)
        )

        acceptable_exceedance = values > rule["acceptable"]

        conditional_limit = pd.Series(
            np.nan, index=model_df.index
        )

        conditional_limit.loc[
            magnesium_valid & (magnesium <= 30)
        ] = rule["permissible_if_magnesium_le_30"]

        conditional_limit.loc[
            magnesium_valid & (magnesium > 30)
        ] = rule["permissible_if_magnesium_gt_30"]

        permissible_exceedance = (
            values > conditional_limit
        )

        unresolved_permissible = (
            valid_mask & ~magnesium_valid
        )

        permissible_valid_mask = (
            valid_mask & magnesium_valid
        )

    elif rule["type"] == "no_relaxation":

        valid_mask &= values >= 0

        acceptable_exceedance = (
            values > rule["acceptable"]
        )

        permissible_exceedance = acceptable_exceedance

        unresolved_permissible = pd.Series(
            False, index=model_df.index
        )

    else:

        valid_mask &= values >= 0

        acceptable_exceedance = (
            values > rule["acceptable"]
        )

        permissible_exceedance = (
            values > rule["permissible"]
        )

        unresolved_permissible = pd.Series(
            False, index=model_df.index
        )

    acceptable_label = pd.Series(
        pd.NA, index=model_df.index, dtype="Int64"
    )

    permissible_label = pd.Series(
        pd.NA, index=model_df.index, dtype="Int64"
    )

    acceptable_label.loc[valid_mask] = (
        acceptable_exceedance.loc[valid_mask].astype(int)
    )

    if rule["type"] == "sulphate_conditional":

        permissible_label.loc[permissible_valid_mask] = (
            permissible_exceedance.loc[
                permissible_valid_mask
            ].astype(int)
        )

    else:

        permissible_label.loc[valid_mask] = (
            permissible_exceedance.loc[valid_mask].astype(int)
        )

    threshold_labels[
        f"{column}__exceeds_acceptable"
    ] = acceptable_label

    threshold_labels[
        f"{column}__exceeds_permissible"
    ] = permissible_label

    threshold_labels[
        f"{column}__reported_zero"
    ] = values.eq(0).astype("Int64").where(valid_mask)

# Attach labels to the separate modelling working copy.
model_df = pd.concat(
    [model_df, threshold_labels],
    axis=1
)

print("PRELIMINARY THRESHOLD LABELS CREATED")
print("Threshold parameters:", len(threshold_rules))
print("Label columns:", len(threshold_labels.columns))
print("Original processed CSV modified: No")

PRELIMINARY THRESHOLD LABELS CREATED
Threshold parameters: 15
Label columns: 45
Original processed CSV modified: No


In [7]:
threshold_class_summary = []

for column in threshold_rules:

    acceptable_label = threshold_labels[
        f"{column}__exceeds_acceptable"
    ]

    permissible_label = threshold_labels[
        f"{column}__exceeds_permissible"
    ]

    zero_label = threshold_labels[
        f"{column}__reported_zero"
    ]

    valid_count = int(acceptable_label.notna().sum())

    acceptable_exceedances = int(
        acceptable_label.sum(skipna=True)
    )

    permissible_exceedances = int(
        permissible_label.sum(skipna=True)
    )

    permissible_evaluable = int(
        permissible_label.notna().sum()
    )

    threshold_class_summary.append({
        "Parameter": column,
        "Available_Observations": valid_count,
        "Reported_Zeros": int(zero_label.sum(skipna=True)),
        "Exceeds_Acceptable": acceptable_exceedances,
        "Acceptable_Exceedance_%": round(
            acceptable_exceedances / valid_count * 100, 3
        ) if valid_count else np.nan,
        "Exceeds_Permissible": permissible_exceedances,
        "Permissible_Exceedance_%": round(
            permissible_exceedances / permissible_evaluable * 100, 3
        ) if permissible_evaluable else np.nan,
        "Permissibility_Not_Evaluable": (
            valid_count - permissible_evaluable
        )
    })

threshold_class_summary_df = pd.DataFrame(
    threshold_class_summary
)

display(
    threshold_class_summary_df.sort_values(
        "Available_Observations",
        ascending=False
    ).reset_index(drop=True)
)

print("\nTOTAL PRELIMINARY LABEL COLUMNS:",
      len(threshold_labels.columns))

,Parameter,Available_Observations,Reported_Zeros,Exceeds_Acceptable,Acceptable_Exceedance_%,Exceeds_Permissible,Permissible_Exceedance_%,Permissibility_Not_Evaluable
0,Potential of Hydrogen (pH),162719,0,10399,6.391,10399,6.391,0
1,Chloride (mg/L),162033,10,29757,18.365,5008,3.091,0
2,Magnesium (mg/L),150412,162,70219,46.684,11423,7.594,0
3,Calcium (mg/L),149855,76,38451,25.659,4622,3.084,0
4,Total Hardness (mgCaCO3/L),144408,151,88597,61.352,14789,10.241,0
5,Sulphate (mg/L),126326,5806,13802,10.926,12776,10.133,246
6,Total Alkalinity (mg/L as CaCO3),79936,3386,42096,52.662,2751,3.442,0
7,Total Dissolved Solids (mg/L),30151,6646,13007,43.140,1998,6.627,0
8,Arsenic (mg/L),9927,9781,57,0.574,57,0.574,0
9,Cadmium (mg/L),797,795,0,0.000,0,0.000,0



TOTAL PRELIMINARY LABEL COLUMNS: 45


In [8]:
# PHASE 5 — BATCH 2: ADDITIONAL BIS THRESHOLD RULES

uranium_col = "Uranium(mg/L)"
iron_col = "Iron(mg/L)"
manganese_col = "Manganese (mg/L)"

def nullable_binary(values, valid_mask, condition):
    result = pd.Series(
        pd.NA,
        index=model_df.index,
        dtype="Int64"
    )

    result.loc[valid_mask] = (
        condition.loc[valid_mask].astype(int)
    )

    return result


# 1. URANIUM: 0.03 mg/L, no relaxation

uranium = model_df[uranium_col]

uranium_valid = (
    uranium.notna()
    & np.isfinite(uranium)
    & (uranium >= 0)
)

uranium_exceedance = uranium > 0.03

uranium_labels = pd.DataFrame(index=model_df.index)

uranium_labels[
    "Uranium__exceeds_acceptable"
] = nullable_binary(
    uranium, uranium_valid, uranium_exceedance
)

uranium_labels[
    "Uranium__exceeds_permissible"
] = nullable_binary(
    uranium, uranium_valid, uranium_exceedance
)

uranium_labels[
    "Uranium__reported_zero"
] = nullable_binary(
    uranium, uranium_valid, uranium.eq(0)
)


# 2. IRON AND MANGANESE INDIVIDUAL LIMITS

iron = model_df[iron_col]
manganese = model_df[manganese_col]

iron_valid = (
    iron.notna()
    & np.isfinite(iron)
    & (iron >= 0)
)

manganese_valid = (
    manganese.notna()
    & np.isfinite(manganese)
    & (manganese >= 0)
)

iron_labels = pd.DataFrame(index=model_df.index)

# Iron: 0.3 mg/L, no relaxation.
iron_exceedance = iron > 0.3

iron_labels[
    "Iron__exceeds_acceptable"
] = nullable_binary(
    iron, iron_valid, iron_exceedance
)

iron_labels[
    "Iron__exceeds_permissible"
] = nullable_binary(
    iron, iron_valid, iron_exceedance
)

# Manganese: 0.1 acceptable, 0.3 permissible.
manganese_labels = pd.DataFrame(index=model_df.index)

manganese_labels[
    "Manganese__exceeds_acceptable"
] = nullable_binary(
    manganese,
    manganese_valid,
    manganese > 0.1
)

manganese_labels[
    "Manganese__exceeds_permissible"
] = nullable_binary(
    manganese,
    manganese_valid,
    manganese > 0.3
)


# 3. JOINT IRON + MANGANESE CONDITION

joint_valid = iron_valid & manganese_valid

combined_fe_mn = iron + manganese

joint_exceedance = combined_fe_mn > 0.3

joint_labels = pd.DataFrame(index=model_df.index)

joint_labels[
    "Iron_Manganese__combined_exceeds_0_3"
] = nullable_binary(
    combined_fe_mn,
    joint_valid,
    joint_exceedance
)


# 4. ATTACH ALL NEW LABELS

additional_labels = pd.concat(
    [
        uranium_labels,
        iron_labels,
        manganese_labels,
        joint_labels
    ],
    axis=1
)

for column in additional_labels.columns:
    threshold_labels[column] = additional_labels[column]
    model_df[column] = additional_labels[column]


# 5. SUMMARY

additional_summary = pd.DataFrame([
    {
        "Parameter": uranium_col,
        "Available": int(uranium_valid.sum()),
        "Reported_Zeros": int((uranium_valid & uranium.eq(0)).sum()),
        "Above_Acceptable": int(
            uranium_exceedance[uranium_valid].sum()
        ),
        "Above_Permissible": int(
            uranium_exceedance[uranium_valid].sum()
        )
    },
    {
        "Parameter": iron_col,
        "Available": int(iron_valid.sum()),
        "Reported_Zeros": int((iron_valid & iron.eq(0)).sum()),
        "Above_Acceptable": int(
            iron_exceedance[iron_valid].sum()
        ),
        "Above_Permissible": int(
            iron_exceedance[iron_valid].sum()
        )
    },
    {
        "Parameter": manganese_col,
        "Available": int(manganese_valid.sum()),
        "Reported_Zeros": int(
            (manganese_valid & manganese.eq(0)).sum()
        ),
        "Above_Acceptable": int(
            (manganese[manganese_valid] > 0.1).sum()
        ),
        "Above_Permissible": int(
            (manganese[manganese_valid] > 0.3).sum()
        )
    }
])

joint_summary = pd.DataFrame([{
    "Rule": "Iron + Manganese combined concentration",
    "Paired_Observations": int(joint_valid.sum()),
    "Exceeds_0_3_mg_L": int(
        joint_exceedance[joint_valid].sum()
    )
}])

print("ADDITIONAL TARGET SUMMARY")
display(additional_summary)

print("\nJOINT IRON-MANGANESE RULE")
display(joint_summary)

print("\nAdditional label columns created:",
      len(additional_labels.columns))

print("Original processed CSV modified: No")

ADDITIONAL TARGET SUMMARY


C:\Users\pokep\AppData\Local\Temp\ipykernel_30236\1811820683.py:141: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  model_df[column] = additional_labels[column]


,Parameter,Available,Reported_Zeros,Above_Acceptable,Above_Permissible
0,Uranium(mg/L),5904,3356,1915,1915
1,Iron(mg/L),38694,9589,13740,13740
2,Manganese (mg/L),797,365,91,40



JOINT IRON-MANGANESE RULE


,Rule,Paired_Observations,Exceeds_0_3_mg_L
0,Iron + Manganese combined concentration,788,106



Additional label columns created: 8
Original processed CSV modified: No


In [9]:
# PRIMARY REGRESSION TARGETS
primary_regression_targets = [
    "Potential of Hydrogen (pH)",
    "Electric Conductivity (μS/cm)",
    "Chloride (mg/L)",
    "Bicarbonate (mg/L)",
    "Sulphate (mg/L)",
    "Calcium (mg/L)",
    "Magnesium (mg/L)",
    "Sodium (mg/L)"
]

# Additional regression candidate
secondary_regression_targets = [
    "Total Hardness (mgCaCO3/L)",
    "Total Alkalinity (mg/L as CaCO3)",
    "Carbonate (mg/L)",
    "Total Dissolved Solids (mg/L)",
    "Iron(mg/L)",
    "Silica(mg/L)"
]

# Primary drinking-water screening classification candidates
primary_classification_parameters = [
    "Potential of Hydrogen (pH)",
    "Chloride (mg/L)",
    "Magnesium (mg/L)",
    "Calcium (mg/L)",
    "Total Hardness (mgCaCO3/L)",
    "Sulphate (mg/L)",
    "Total Alkalinity (mg/L as CaCO3)"
]

# Secondary / specialized classification candidates
secondary_classification_parameters = [
    "Total Dissolved Solids (mg/L)",
    "Iron(mg/L)"
]

# Exploratory targets requiring additional validation
exploratory_classification_parameters = [
    "Arsenic (mg/L)",
    "Uranium(mg/L)",
    "Manganese (mg/L)",
    "Copper (mg/L)",
    "Lead (mg/L)",
    "Zinc (mg/L)",
    "Nickel (mg/L)",
    "Cadmium (mg/L)",
    "Chromium (mg/L)"
]

target_plan = pd.DataFrame([
    {
        "Parameter": column,
        "Regression_Role": (
            "Primary"
            if column in primary_regression_targets
            else "Secondary"
            if column in secondary_regression_targets
            else "Not selected"
        ),
        "Classification_Role": (
            "Primary"
            if column in primary_classification_parameters
            else "Secondary"
            if column in secondary_classification_parameters
            else "Exploratory"
            if column in exploratory_classification_parameters
            else "No documented threshold"
        )
    }
    for column in chemical_columns
])

print("MODELLING TARGET REGISTER")
display(target_plan)

print("\nPRIMARY MULTI-OUTPUT REGRESSION TARGETS:",
      len(primary_regression_targets))

print("PRIMARY CLASSIFICATION TARGETS:",
      len(primary_classification_parameters))

MODELLING TARGET REGISTER


,Parameter,Regression_Role,Classification_Role
0,Potential of Hydrogen (pH),Primary,Primary
1,Electric Conductivity (μS/cm),Primary,No documented threshold
2,Total Dissolved Solids (mg/L),Secondary,Secondary
3,Carbonate (mg/L),Secondary,No documented threshold
4,Bicarbonate (mg/L),Primary,No documented threshold
5,Total Alkalinity (mg/L as CaCO3),Secondary,Primary
6,Chloride (mg/L),Primary,Primary
7,Nitrate N (mgN/L),Not selected,No documented threshold
8,Sulphate (mg/L),Primary,Primary
9,Phosphate(mg/L),Not selected,No documented threshold



PRIMARY MULTI-OUTPUT REGRESSION TARGETS: 8
PRIMARY CLASSIFICATION TARGETS: 7


In [10]:
feature_df = pd.DataFrame(index=model_df.index)

feature_df["_year"] = model_df["_parsed_date"].dt.year
feature_df["_month"] = model_df["_parsed_date"].dt.month
feature_df["_quarter"] = model_df["_parsed_date"].dt.quarter
feature_df["_day_of_year"] = model_df["_parsed_date"].dt.dayofyear

# Numeric year trend
feature_df["_year_index"] = feature_df["_year"] - 2000

# Cyclical month encoding
feature_df["_month_sin"] = np.sin(
    2 * np.pi * feature_df["_month"] / 12
)

feature_df["_month_cos"] = np.cos(
    2 * np.pi * feature_df["_month"] / 12
)

# Cyclical day-of-year encoding
feature_df["_dayofyear_sin"] = np.sin(
    2 * np.pi * feature_df["_day_of_year"] / 365.25
)

feature_df["_dayofyear_cos"] = np.cos(
    2 * np.pi * feature_df["_day_of_year"] / 365.25
)

print("TEMPORAL FEATURES CREATED")
display(
    feature_df[
        [
            "_year",
            "_month",
            "_quarter",
            "_year_index",
            "_month_sin",
            "_month_cos",
            "_dayofyear_sin",
            "_dayofyear_cos"
        ]
    ].head()
)

print("\nMissing temporal feature values:")
display(feature_df.isna().sum())

TEMPORAL FEATURES CREATED


C:\Users\pokep\AppData\Local\Temp\ipykernel_30236\1930772485.py:6: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  feature_df["_day_of_year"] = model_df["_parsed_date"].dt.dayofyear


,_year,_month,_quarter,_year_index,_month_sin,_month_cos,_dayofyear_sin,_dayofyear_cos
0,2004,5,2,4,0.5,-0.866025,0.589176,-0.808005
1,2005,5,2,5,0.5,-0.866025,0.602988,-0.797750
2,2007,5,2,7,0.5,-0.866025,0.602988,-0.797750
3,2009,5,2,9,0.5,-0.866025,0.602988,-0.797750
4,2010,5,2,10,0.5,-0.866025,0.602988,-0.797750



Missing temporal feature values:


_year             0
_month            0
_quarter          0
_day_of_year      0
_year_index       0
_month_sin        0
_month_cos        0
_dayofyear_sin    0
_dayofyear_cos    0
dtype: int64

In [11]:
# Normalized state and district categories

feature_df["_state_feature"] = (
    model_df["_state_norm"]
    .astype("string")
    .str.strip()
)

feature_df["_district_feature"] = (
    model_df["_district_norm"]
    .astype("string")
    .str.strip()
)

feature_df["_state_district_feature"] = (
    feature_df["_state_feature"]
    + "|"
    + feature_df["_district_feature"]
)

# Numeric geographic coordinates
feature_df["_latitude"] = model_df["Latitude"]
feature_df["_longitude"] = model_df["Longitude"]

# 0.1-degree spatial cells for grouping and spatial validation.
# These are not used as baseline predictive features.

feature_df["_latitude_cell"] = (
    np.floor(feature_df["_latitude"] * 10) / 10
)

feature_df["_longitude_cell"] = (
    np.floor(feature_df["_longitude"] * 10) / 10
)

feature_df["_geo_grid_id"] = (
    feature_df["_latitude_cell"].astype("string")
    + "|"
    + feature_df["_longitude_cell"].astype("string")
)

# Location grouping identifier:
# Prefer the coordinate-based candidate; otherwise use provisional station ID.

location_candidate = (
    model_df["_provisional_location_candidate_id"]
    .replace("", np.nan)
)

feature_df["_spatial_group_id"] = (
    location_candidate
    .fillna(model_df["_provisional_station_id"])
)

print("GEOGRAPHICAL FEATURES CREATED")
print("=" * 50)

print("Unique states:",
      feature_df["_state_feature"].nunique())

print("Unique state-district categories:",
      feature_df["_state_district_feature"].nunique())

print("Unique spatial grid cells:",
      feature_df["_geo_grid_id"].nunique())

print("Unique spatial grouping identifiers:",
      feature_df["_spatial_group_id"].nunique())

print("\nFEATURE PREVIEW")
display(
    feature_df[
        [
            "_latitude",
            "_longitude",
            "_state_feature",
            "_district_feature",
            "_state_district_feature",
            "_geo_grid_id"
        ]
    ].head()
)

GEOGRAPHICAL FEATURES CREATED
Unique states: 31
Unique state-district categories: 598
Unique spatial grid cells: 12810
Unique spatial grouping identifiers: 24199

FEATURE PREVIEW


,_latitude,_longitude,_state_feature,_district_feature,_state_district_feature,_geo_grid_id
0,11.708611,92.65,andaman and nicobar islands,south andaman,andaman and nicobar islands|south andaman,11.7|92.6
1,11.708611,92.65,andaman and nicobar islands,south andaman,andaman and nicobar islands|south andaman,11.7|92.6
2,11.708611,92.65,andaman and nicobar islands,south andaman,andaman and nicobar islands|south andaman,11.7|92.6
3,11.708611,92.65,andaman and nicobar islands,south andaman,andaman and nicobar islands|south andaman,11.7|92.6
4,11.708611,92.65,andaman and nicobar islands,south andaman,andaman and nicobar islands|south andaman,11.7|92.6


In [12]:
# Context-only baseline predictors
baseline_feature_columns = [
    "_latitude",
    "_longitude",
    "_state_feature",
    "_district_feature",
    "_state_district_feature",
    "_year_index",
    "_month_sin",
    "_month_cos",
    "_dayofyear_sin",
    "_dayofyear_cos"
]

X_context = feature_df[
    baseline_feature_columns
].copy()

# Primary eight-output regression target matrix
Y_core = model_df[
    primary_regression_targets
].copy()

# Apply basic target screening without overwriting source measurements.
for column in primary_regression_targets:

    if column == "Potential of Hydrogen (pH)":

        Y_core.loc[
            ~Y_core[column].between(0.000001, 14),
            column
        ] = np.nan

    else:

        Y_core.loc[
            Y_core[column] < 0,
            column
        ] = np.nan

# Complete observations for all eight regression targets
core_complete_mask = Y_core.notna().all(axis=1)

X_core_complete = X_context.loc[
    core_complete_mask
].copy()

Y_core_complete = Y_core.loc[
    core_complete_mask
].copy()

# Compare with an extended nine-target dataset including hardness.
extended_targets = (
    primary_regression_targets
    + ["Total Hardness (mgCaCO3/L)"]
)

Y_extended = model_df[extended_targets].copy()

for column in extended_targets:

    if column == "Potential of Hydrogen (pH)":
        Y_extended.loc[
            ~Y_extended[column].between(0.000001, 14),
            column
        ] = np.nan

    else:
        Y_extended.loc[
            Y_extended[column] < 0,
            column
        ] = np.nan

extended_complete_count = int(
    Y_extended.notna().all(axis=1).sum()
)

print("MODELLING MATRIX READINESS")
print("=" * 55)

print("Total observations:", len(model_df))
print("Context predictor columns:", len(baseline_feature_columns))
print("Primary regression targets:", len(primary_regression_targets))

print("\nComplete observations for eight targets:",
      len(Y_core_complete))

print("Complete observations for nine targets:",
      extended_complete_count)

print("\nBASELINE PREDICTOR MISSINGNESS")
display(X_context.isna().sum().rename("Missing_Values"))

# Check that baseline predictors contain no chemical target columns
chemical_leakage = [
    col for col in baseline_feature_columns
    if col in chemical_columns
]

identifier_leakage = [
    col for col in baseline_feature_columns
    if col in [
        "SlNo",
        "Station",
        "_record_id",
        "_source_row",
        "_source_file",
        "_provisional_station_id",
        "_provisional_location_candidate_id"
    ]
]

print("\nLEAKAGE CHECK")
print("Chemical target columns in predictors:", chemical_leakage)
print("Record or station identifiers in predictors:", identifier_leakage)

print("\nGROUPING FIELDS RESERVED FOR VALIDATION")
print("Spatial groups:", feature_df["_spatial_group_id"].nunique())

assert len(X_core_complete) == len(Y_core_complete)
assert len(chemical_leakage) == 0
assert len(identifier_leakage) == 0

print("\nBASELINE MODELLING MATRICES CREATED SUCCESSFULLY.")

MODELLING MATRIX READINESS
Total observations: 174635
Context predictor columns: 10
Primary regression targets: 8

Complete observations for eight targets: 122713
Complete observations for nine targets: 121361

BASELINE PREDICTOR MISSINGNESS


_latitude                  152
_longitude                 152
_state_feature               0
_district_feature            0
_state_district_feature      0
_year_index                  0
_month_sin                   0
_month_cos                   0
_dayofyear_sin               0
_dayofyear_cos               0
Name: Missing_Values, dtype: int64


LEAKAGE CHECK
Chemical target columns in predictors: []
Record or station identifiers in predictors: []

GROUPING FIELDS RESERVED FOR VALIDATION
Spatial groups: 24199

BASELINE MODELLING MATRICES CREATED SUCCESSFULLY.


In [13]:
# Add an explicit indicator for missing coordinates
feature_df["_coordinates_missing"] = (
    feature_df["_latitude"].isna()
    | feature_df["_longitude"].isna()
).astype(int)

# Final baseline context predictors
baseline_feature_columns = [
    "_latitude",
    "_longitude",
    "_coordinates_missing",
    "_state_feature",
    "_district_feature",
    "_state_district_feature",
    "_year_index",
    "_month_sin",
    "_month_cos",
    "_dayofyear_sin",
    "_dayofyear_cos"
]

X_context = feature_df[
    baseline_feature_columns
].copy()

# Target selection register
target_plan["Modelling_Priority"] = "Not selected"

target_plan.loc[
    target_plan["Parameter"].isin(primary_regression_targets),
    "Modelling_Priority"
] = "Primary regression"

target_plan.loc[
    target_plan["Parameter"].isin(secondary_regression_targets),
    "Modelling_Priority"
] = "Secondary regression"

target_plan.loc[
    target_plan["Parameter"].isin(primary_classification_parameters),
    "Modelling_Priority"
] = "Primary classification"

target_plan.loc[
    target_plan["Parameter"].isin(secondary_classification_parameters),
    "Modelling_Priority"
] = "Secondary classification"

target_plan.loc[
    target_plan["Parameter"].isin(exploratory_classification_parameters),
    "Modelling_Priority"
] = "Exploratory classification"

print("FINAL FEATURE REGISTER")
print("Predictor columns:", len(baseline_feature_columns))
display(pd.DataFrame({
    "Feature": baseline_feature_columns,
    "Data_Type": X_context.dtypes.astype(str).values,
    "Missing_Values": X_context.isna().sum().values
}))

print("\nFINAL TARGET SELECTION REGISTER")
display(target_plan)

FINAL FEATURE REGISTER
Predictor columns: 11


,Feature,Data_Type,Missing_Values
0,_latitude,float64,152
1,_longitude,float64,152
2,_coordinates_missing,int64,0
3,_state_feature,string,0
4,_district_feature,string,0
5,_state_district_feature,string,0
6,_year_index,int32,0
7,_month_sin,float64,0
8,_month_cos,float64,0
9,_dayofyear_sin,float64,0



FINAL TARGET SELECTION REGISTER


,Parameter,Regression_Role,Classification_Role,Modelling_Priority
0,Potential of Hydrogen (pH),Primary,Primary,Primary classification
1,Electric Conductivity (μS/cm),Primary,No documented threshold,Primary regression
2,Total Dissolved Solids (mg/L),Secondary,Secondary,Secondary classification
3,Carbonate (mg/L),Secondary,No documented threshold,Secondary regression
4,Bicarbonate (mg/L),Primary,No documented threshold,Primary regression
5,Total Alkalinity (mg/L as CaCO3),Secondary,Primary,Primary classification
6,Chloride (mg/L),Primary,Primary,Primary classification
7,Nitrate N (mgN/L),Not selected,No documented threshold,Not selected
8,Sulphate (mg/L),Primary,Primary,Primary classification
9,Phosphate(mg/L),Not selected,No documented threshold,Not selected


In [16]:
# PHASE 5 — BATCH 4: REGRESSION DATASET PREPARATION

# Source metadata available in model_df
metadata_columns = [
    "_record_id",
    "_source_file",
    "_source_row",
    "_parsed_date",
    "_provisional_station_id",
    "_provisional_location_candidate_id"
]

# Combine metadata with engineered grouping features
regression_metadata = pd.concat(
    [
        model_df[metadata_columns],
        feature_df[
            ["_spatial_group_id", "_geo_grid_id"]
        ]
    ],
    axis=1
)

# Prepare primary eight-target regression matrix
Y_core = model_df[primary_regression_targets].copy()

for column in primary_regression_targets:

    if column == "Potential of Hydrogen (pH)":
        Y_core.loc[
            ~Y_core[column].between(0.000001, 14),
            column
        ] = np.nan
    else:
        Y_core.loc[
            Y_core[column] < 0,
            column
        ] = np.nan

core_mask = Y_core.notna().all(axis=1)

regression_core_dataset = pd.concat(
    [
        regression_metadata.loc[core_mask],
        X_context.loc[core_mask],
        Y_core.loc[core_mask]
    ],
    axis=1
)

# Prepare extended nine-target regression matrix
extended_targets = (
    primary_regression_targets
    + ["Total Hardness (mgCaCO3/L)"]
)

Y_extended = model_df[extended_targets].copy()

for column in extended_targets:

    if column == "Potential of Hydrogen (pH)":
        Y_extended.loc[
            ~Y_extended[column].between(0.000001, 14),
            column
        ] = np.nan
    else:
        Y_extended.loc[
            Y_extended[column] < 0,
            column
        ] = np.nan

extended_mask = Y_extended.notna().all(axis=1)

regression_extended_dataset = pd.concat(
    [
        regression_metadata.loc[extended_mask],
        X_context.loc[extended_mask],
        Y_extended.loc[extended_mask]
    ],
    axis=1
)

print("REGRESSION DATASET PREPARATION")
print("=" * 55)

print("Eight-target observations:", len(regression_core_dataset))
print("Nine-target observations:", len(regression_extended_dataset))

print("\nEight-target output columns:", len(primary_regression_targets))
print("Nine-target output columns:", len(extended_targets))

print("\nMissing coordinate records in eight-target dataset:",
      regression_core_dataset["_coordinates_missing"].sum())

print("\nPrimary regression dataset dimensions:",
      regression_core_dataset.shape)

print("Extended regression dataset dimensions:",
      regression_extended_dataset.shape)

REGRESSION DATASET PREPARATION
Eight-target observations: 122713
Nine-target observations: 121361

Eight-target output columns: 8
Nine-target output columns: 9

Missing coordinate records in eight-target dataset: 98

Primary regression dataset dimensions: (122713, 27)
Extended regression dataset dimensions: (121361, 28)


In [17]:
# Select parameters for which threshold labels were prepared
classification_parameters = (
    primary_classification_parameters
    + secondary_classification_parameters
    + exploratory_classification_parameters
    + [
        "Uranium(mg/L)",
        "Manganese (mg/L)"
    ]
)

# Remove accidental duplicate parameter names while preserving order
classification_parameters = list(
    dict.fromkeys(classification_parameters)
)

classification_label_columns = []

for parameter in classification_parameters:

    for suffix in [
        "__exceeds_acceptable",
        "__exceeds_permissible"
    ]:

        label_column = parameter + suffix

        if label_column in model_df.columns:
            classification_label_columns.append(label_column)

# Include the joint iron-manganese screening label
joint_label = "Iron_Manganese__combined_exceeds_0_3"

if joint_label in model_df.columns:
    classification_label_columns.append(joint_label)

# One row per original observation, with separate target-label columns
classification_dataset = pd.concat(
    [
        regression_metadata,
        X_context,
        model_df[classification_label_columns]
    ],
    axis=1
)

# Assess each target's available labels and positive cases
classification_support_rows = []

for column in classification_label_columns:

    labels = pd.to_numeric(
        classification_dataset[column],
        errors="coerce"
    )

    valid_labels = labels.dropna()

    positive_count = int((valid_labels == 1).sum())
    negative_count = int((valid_labels == 0).sum())

    classification_support_rows.append({
        "Label": column,
        "Available_Labels": len(valid_labels),
        "Positive_Count": positive_count,
        "Negative_Count": negative_count,
        "Positive_Class_Percentage": round(
            positive_count / len(valid_labels) * 100, 3
        ) if len(valid_labels) else np.nan
    })

classification_support_df = pd.DataFrame(
    classification_support_rows
)

print("CLASSIFICATION DATASET")
print("=" * 55)
print("Total records:", len(classification_dataset))
print("Classification target columns:",
      len(classification_label_columns))

print("\nCLASSIFICATION TARGET SUPPORT")
display(
    classification_support_df.sort_values(
        "Available_Labels",
        ascending=False
    ).reset_index(drop=True)
)

print("\nNote: Missing labels are retained as missing, not converted to zero.")

CLASSIFICATION DATASET
Total records: 174635
Classification target columns: 31

CLASSIFICATION TARGET SUPPORT


,Label,Available_Labels,Positive_Count,Negative_Count,Positive_Class_Percentage
0,Potential of Hydrogen (pH)__exceeds_acceptable,162719,10399,152320,6.391
1,Potential of Hydrogen (pH)__exceeds_permissible,162719,10399,152320,6.391
2,Chloride (mg/L)__exceeds_acceptable,162033,29757,132276,18.365
3,Chloride (mg/L)__exceeds_permissible,162033,5008,157025,3.091
4,Magnesium (mg/L)__exceeds_acceptable,150412,70219,80193,46.684
5,Magnesium (mg/L)__exceeds_permissible,150412,11423,138989,7.594
6,Calcium (mg/L)__exceeds_acceptable,149855,38451,111404,25.659
7,Calcium (mg/L)__exceeds_permissible,149855,4622,145233,3.084
8,Total Hardness (mgCaCO3/L)__exceeds_acceptable,144408,88597,55811,61.352
9,Total Hardness (mgCaCO3/L)__exceeds_permissible,144408,14789,129619,10.241



Note: Missing labels are retained as missing, not converted to zero.


In [18]:
output_files = {
    "regression_core": FEATURE_DIR / "regression_core_8_targets.csv",
    "regression_extended": FEATURE_DIR / "regression_extended_9_targets.csv",
    "classification": FEATURE_DIR / "classification_context_labels.csv",
    "target_register": FEATURE_DIR / "target_selection_register.csv",
    "classification_support": FEATURE_DIR / "classification_label_support.csv",
    "threshold_register": FEATURE_DIR / "threshold_reference_register.csv",
    "target_feasibility": FEATURE_DIR / "target_feasibility_assessment.csv"
}

# Export modelling datasets
regression_core_dataset.to_csv(
    output_files["regression_core"],
    index=False,
    encoding="utf-8-sig"
)

regression_extended_dataset.to_csv(
    output_files["regression_extended"],
    index=False,
    encoding="utf-8-sig"
)

classification_dataset.to_csv(
    output_files["classification"],
    index=False,
    encoding="utf-8-sig"
)

# Export supporting reports
target_plan.to_csv(
    output_files["target_register"],
    index=False,
    encoding="utf-8-sig"
)

classification_support_df.to_csv(
    output_files["classification_support"],
    index=False,
    encoding="utf-8-sig"
)

threshold_registry.to_csv(
    output_files["threshold_register"],
    index=False,
    encoding="utf-8-sig"
)

target_feasibility_df.to_csv(
    output_files["target_feasibility"],
    index=False,
    encoding="utf-8-sig"
)

# Final integrity validation
print("PHASE 5 — FINAL EXPORT VALIDATION")
print("=" * 60)

for name, path in output_files.items():
    print(
        name,
        "| Exists:", path.exists(),
        "| Size:",
        round(path.stat().st_size / (1024 ** 2), 2),
        "MB"
    )

print("\nDATASET VALIDATION")

print("Primary regression records:",
      len(regression_core_dataset))

print("Extended regression records:",
      len(regression_extended_dataset))

print("Classification records:",
      len(classification_dataset))

print("Unique primary regression IDs:",
      regression_core_dataset["_record_id"].nunique())

print("Unique extended regression IDs:",
      regression_extended_dataset["_record_id"].nunique())

print("Unique classification IDs:",
      classification_dataset["_record_id"].nunique())

print("\nLABEL VALIDATION")

label_values = classification_dataset[
    classification_label_columns
]

invalid_label_values = (
    label_values.notna()
    & ~label_values.isin([0, 1])
).sum().sum()

print("Classification label columns:",
      len(classification_label_columns))

print("Invalid classification label values:",
      invalid_label_values)

print("Missing classification labels:",
      label_values.isna().sum().sum())

# Assertions
assert len(regression_core_dataset) == 122713
assert len(regression_extended_dataset) == 121361
assert len(classification_dataset) == 174635

assert regression_core_dataset["_record_id"].is_unique
assert regression_extended_dataset["_record_id"].is_unique
assert classification_dataset["_record_id"].is_unique

assert invalid_label_values == 0
assert all(path.exists() for path in output_files.values())

assert regression_core_dataset[primary_regression_targets].notna().all().all()
assert regression_extended_dataset[extended_targets].notna().all().all()

print("\nALL VALIDATION CHECKS PASSED")
print("PHASE 5 — FEATURE ENGINEERING & TARGET PREPARATION COMPLETED")

PHASE 5 — FINAL EXPORT VALIDATION
regression_core | Exists: True | Size: 54.1 MB
regression_extended | Exists: True | Size: 54.18 MB
classification | Exists: True | Size: 76.52 MB
target_register | Exists: True | Size: 0.0 MB
classification_support | Exists: True | Size: 0.0 MB
threshold_register | Exists: True | Size: 0.0 MB
target_feasibility | Exists: True | Size: 0.0 MB

DATASET VALIDATION
Primary regression records: 122713
Extended regression records: 121361
Classification records: 174635
Unique primary regression IDs: 122713
Unique extended regression IDs: 121361
Unique classification IDs: 174635

LABEL VALIDATION
Classification label columns: 31
Invalid classification label values: 0
Missing classification labels: 3372055

ALL VALIDATION CHECKS PASSED
PHASE 5 — FEATURE ENGINEERING & TARGET PREPARATION COMPLETED



# PHASE 5 — FINAL COMPLETION REPORT
## Feature Engineering & Target Preparation

**Project:** Borewell Sentinel India  
**Phase:** 5  
**Status:** COMPLETED

---

## 1. Phase Overview

Phase 5 successfully established the prediction targets, threshold-based classification labels and engineered predictor features required for subsequent machine learning development.

The integrated processed dataset contains 174,635 observations.

Three separate modelling datasets were successfully prepared.

## 2. Prediction Target Selection

### Primary Regression Targets

Eight chemical parameters were selected as the primary multi-output regression candidates:

1. Potential of Hydrogen (pH)
2. Electrical Conductivity
3. Chloride
4. Bicarbonate
5. Sulphate
6. Calcium
7. Magnesium
8. Sodium

Total Hardness was included in a separate extended regression dataset.

### Secondary Regression Candidates

- Total Hardness
- Total Alkalinity
- Carbonate
- Total Dissolved Solids
- Iron
- Silica

### Classification Candidates

Primary and secondary classification candidates were selected based on available measurements and documented screening thresholds.

Trace-metal classification targets were retained for exploratory or specialized modelling because their observation counts and positive-class frequencies are limited.

Nitrate, fluoride and potassium could not be prepared as prediction targets because the source dataset contains no recorded measurements for these parameters.

## 3. Threshold-Based Classification Preparation

A threshold reference register was established using the BIS drinking-water screening framework.

The classification preparation includes:

- Acceptable-limit exceedance labels.
- Permissible-limit exceedance labels.
- Uranium screening labels.
- Individual iron and manganese labels.
- Combined iron-manganese screening label.

The classification labels are represented using:

- 1: Screening-limit exceedance.
- 0: No screening-limit exceedance.
- Missing: No usable measurement or insufficient information to evaluate the applicable rule.

These labels represent drinking-water screening benchmark comparisons. They are not automatic declarations of groundwater safety or unsafety.

Further validation of measurement detection limits, analyte definitions and units remains important before interpreting individual trace-metal results.

## 4. Engineered Features

### Geographical Features

- Latitude
- Longitude
- Missing-coordinate indicator
- Normalized state
- Normalized district
- State-district category
- Spatial grid identifier
- Provisional spatial grouping identifier

### Temporal Features

- Year index
- Month
- Quarter
- Day of year
- Cyclical month encoding
- Cyclical day-of-year encoding

### Leakage Prevention

The initial baseline predictor set contains geographical and temporal information only.

The chemical target columns are excluded from the baseline predictors.

Record identifiers, station identifiers and source filenames are retained for traceability and grouping, not used as ordinary baseline predictors.

Future temporal information must not be used to predict past observations.

## 5. Final Modelling Datasets

| Dataset | Records | Prediction Targets |
|---|---:|---:|
| Primary regression | 122,713 | 8 |
| Extended regression | 121,361 | 9 |
| Classification | 174,635 | 31 label columns |

The primary regression dataset contains records with valid values for all eight selected targets.

The extended dataset additionally requires Total Hardness, resulting in fewer complete observations.

The classification dataset retains all original observations and preserves missing labels as missing.

## 6. Classification Label Quality

The final classification dataset contains 31 screening-label columns.

- Invalid label values: 0
- Permitted label values: 0 and 1
- Unavailable or unevaluable labels remain missing.
- No automatic conversion of missing labels to negative cases was performed.

This approach prevents unavailable chemical measurements from being incorrectly treated as clean or non-exceeding observations.

## 7. Exported Outputs

All Phase 5 outputs are stored in:

`reports/feature_engineering/`

| File | Purpose |
|---|---|
| `regression_core_8_targets.csv` | Primary eight-target regression dataset |
| `regression_extended_9_targets.csv` | Extended nine-target regression dataset |
| `classification_context_labels.csv` | Classification dataset with screening labels |
| `target_selection_register.csv` | Target selection and modelling priority register |
| `classification_label_support.csv` | Classification availability and class balance |
| `threshold_reference_register.csv` | Drinking-water threshold reference register |
| `target_feasibility_assessment.csv` | Chemical target feasibility assessment |

## 8. Final Integrity Validation

The final export validation confirmed:

| Validation | Result |
|---|---|
| Primary regression records | 122,713 |
| Extended regression records | 121,361 |
| Classification records | 174,635 |
| Unique primary regression identifiers | 122,713 |
| Unique extended regression identifiers | 121,361 |
| Unique classification identifiers | 174,635 |
| Classification label columns | 31 |
| Invalid classification label values | 0 |
| Export files created | 7 |
| Final validation assertions | Passed |

All seven output files were successfully created.

## 9. Remaining Modelling Considerations

The prepared datasets establish the initial modelling foundation, but several considerations remain:

1. Trace-metal targets have limited observations and class imbalance.
2. Reported zero measurements require cautious interpretation.
3. Spatial grouping must be respected during train/test validation.
4. Temporal evaluation must avoid future-data leakage.
5. BIS screening labels must not be interpreted as definitive safety assessments.
6. Models must be evaluated against appropriate target-specific data availability.
7. More advanced missing-data strategies must be assessed separately rather than applied indiscriminately.

No machine-learning models have been trained or evaluated in this phase.

## 10. Phase Conclusion

Phase 5 successfully transformed the integrated groundwater dataset into structured regression and classification modelling datasets.

Target feasibility, screening-label preparation, geographical and temporal feature engineering, and export validation have been completed.

The project now has a structured foundation for developing its first multi-contaminant prediction models.

**PHASE 5 — FEATURE ENGINEERING & TARGET PREPARATION: COMPLETED**

---

# NEXT PHASE

## PHASE 6 — MULTI-CONTAMINANT PREDICTION

The next phase will focus on building, training and comparing machine learning models for groundwater chemical concentration prediction and screening-limit exceedance classification.
